## 第8章 文件和数据持久化

### 1.文件操作

详细介绍见：[Python基础教程(第3版)-第11章-文件](../03pybeginning/第11章-文件.ipynb)

- 打开文件：`open(文件路径, 模式)`，模式包括：`r`只读，`w`写入(会覆盖文件内容)，`a`追加，`b`二进制，`t`文本，`+`读写，默认是`r`。
- 读取文件：`read()`，返回一个字符串，包含文件的所有内容。`readlines()`，返回一个列表，每个元素是一个文件的行。`readline()`，返回一个字符串，包含文件的一行。
- 写入文件：`write(要写入的内容)`，返回写入的字符数。
- 关闭文件：`close()`


In [ ]:
with open('./res/fear.txt', encoding='utf-8') as f:
    for line in f:
        print(line.strip())

### 2.目录操作

Python中提供`os.path`和`pathlib`模块，用于处理文件系统路径。os.path是对字符串进行操作，pathlib提供了类表示文件系统，可以适应不同的操作系统，因此，**一般建议使用pathlib**。

In [ ]:
from pathlib import Path

# 创建路径对象
p = Path('./res') / 'test.txt'       # 跨平台拼接 ⭐
print(p)                             # res/test.txt

# 路径属性
print(p.name)           # 'test.txt'，文件名
print(p.stem)           # 'test'，文件名(不包含扩展名)
print(p.suffix)         # '.txt'，文件扩展名
print(p.parent)         # res，父目录
print(p.exists())       # True，文件是否存在
print(p.is_file())      # True，是否为文件
print(p.is_dir())       # False，是否为目录

# 读写文本（Path 对象自带方法 ⭐）
content = p.read_text()               # 读取全部文本
p.write_text('New content')           # 写入文本

# 创建目录
Path('./res/newdir').mkdir(exist_ok=True)   # 已存在不报错
Path('./res/a/b/c').mkdir(parents=True)     # 递归创建

# 列出目录内容
print(list(Path('.').iterdir()))            # 当前目录下所有条目
print(list(Path('.').glob('*.ipynb')))      # 匹配 .ipynb 文件
print(list(Path('.').rglob('*.ipynb')))     # 递归匹配 ⭐

# 重命名/移动
p = p.rename('./res/newname.txt')

# 删除
p.unlink()                               # 删除文件
Path('./res/empty').rmdir()              # 删除空目录

可以使用`shutil`模块来操作文件和目录。
- `shutil.copy(src, dst)`：复制文件或目录。
- `shutil.copytree(src, dst)`：递归复制目录树。
- `shutil.move(src, dst)`：移动文件或目录。
- `shutil.rmtree(path)`：删除目录树。

可以使用`tempfile`模块来创建临时文件或目录。
- `tempfile.NamedTemporaryFile()`：创建临时文件。
- `tempfile.TemporaryDirectory()`：创建临时目录。
- `tempfile.mkdtemp()`：创建临时目录(返回目录名)。

In [ ]:
from pathlib import Path
import tempfile

# 临时文件
with tempfile.NamedTemporaryFile(dir='./res', delete=False, mode='w') as f:
    f.write('Temporary content')
    print(f.name)                        # 临时文件路径

# 临时目录，离开with块后，临时目录自动删除
with tempfile.TemporaryDirectory(dir='./res') as tmpdir:
    tmp = Path(tmpdir)
    (tmp / 'file1.txt').write_text('Hello')
    print(list(tmp.iterdir()))

可以是`zipfile`模块对压缩文件进行操作。
- 压缩文件：`zipfile.ZipFile(filename, mode='w', compression=zipfile.ZIP_DEFLATED)`。
    - `filename`：压缩文件名。
    - `mode`：压缩模式，`w`表示写入。
    - `compression`：压缩算法，`ZIP_DEFLATED`表示使用Deflate压缩。
    - 返回值：`ZipFile`对象。
- 解压文件：`zipfile.ZipFile(filename, mode='r')`。
    - `filename`：压缩文件名。
    - `mode`：解压模式，`r`表示读取。
    - 返回值：`ZipFile`对象。

### 3.JSON文件操作

- 序列化：`json.dumps(obj)`，Python → JSON 字符串。`json.dump(obj, f)`，Python → JSON 文件。
- 反序列化：`json.loads(s)`，JSON 字符串 → Python。`json.load(f)`，JSON 文件 → Python。

In [9]:
import json

# 序列化（Python → JSON 字符串）
data = {'name': 'Yoda', 'age': 900, 'jedi': True}
json_str = json.dumps(data)                          # '{"name": "Yoda", "age": 900, "jedi": true}'
json_str_pretty = json.dumps(data, indent=2)         # 缩进格式化 ⭐
json_str_sorted = json.dumps(data, sort_keys=True)   # 键排序

# 反序列化（JSON 字符串 → Python）
parsed = json.loads(json_str)                        # {'name': 'Yoda', 'age': 900, 'jedi': True}

In [10]:
from datetime import datetime
import json

# 自定义 JSON 编码器，处理 datetime 类型
class DatetimeEncoder(json.JSONEncoder):
    def default(self, obj):  # 重写默认方法，处理 datetime 类型
        if isinstance(obj, datetime):
            return obj.isoformat()
        return super().default(obj)       # 其他类型交给父类处理

dt = datetime(2025, 5, 27, 8, 30)
json_str = json.dumps({'timestamp': dt}, cls=DatetimeEncoder)  # 指定自定义编码器
# '{"timestamp": "2025-05-27T08:30:00"}'

# 自定义 JSON 解码器，处理 datetime 类型
def datetime_decoder(dct):
    if 'timestamp' in dct:
        dct['timestamp'] = datetime.fromisoformat(dct['timestamp'])
    return dct

parsed = json.loads(json_str, object_hook=datetime_decoder)  # 指定自定义解码器
# {'timestamp': datetime.datetime(2025, 5, 27, 8, 30)}

### 4.其他操作

`io.StringIO` / `io.BytesIO`模块：用于在内存中进行字符串或字节流的操作。
- `read()`：读取所有数据。
- `write(data)`：写入数据。
- `seek(offset)`：移动位置。
- `tell()`：返回当前位置。

`pickle`模块：用于序列化和反序列化Python对象。pickle能序列化几乎所有Python对象，但是要特别注意：反序列化时会**执行任意代码**，存在安全风险。
- `pickle.dumps(obj)`：将对象序列化为字节流。`pickle.dump(obj, file)`，将对象写入文件。
- `pickle.loads(bytes)`：将字节流反序列化为对象。`pickle.load(file)`，从文件读取对象。

### 5.本章小结

```text
文件与数据持久化
│
├── 文件与目录 ⭐
│   ├── open() + with 上下文管理器
│   ├── 文件模式（r/w/x/a/b/t/+）
│   ├── 读写方法（read/write/readline/readlines/print）
│   ├── pathlib（推荐）vs os.path
│   ├── shutil（复制/移动/删除目录树）
│   └── tempfile（临时文件/目录）
│
├── 压缩
│   ├── zipfile（ZIP 创建/追加/解压）
│   └── tarfile（tar.gz）
│
├── JSON ⭐
│   ├── dumps/loads（内存）vs dump/load（文件）
│   ├── JSON ↔ Python 类型映射
│   ├── 自定义编码器（JSONEncoder.default）
│   └── 自定义解码器（object_hook）
│
├── I/O 流
│   ├── io.StringIO（内存文本流）
│   ├── io.BytesIO（内存二进制流）
│   └── requests（第三方 HTTP ⭐）
│
└── 数据持久化 ⭐
    ├── pickle（Python 专用，二进制，⚠️不安全）
    ├── shelve（字典式持久化，writeback ⚠️）
    └── SQLAlchemy ORM
        ├── 模型定义（Base + Column）
        ├── CRUD（add/query/delete + commit）
        ├── 关系映射（relationship + ForeignKey）
        └── 事务（commit/rollback）
```